In [ ]:
library(numbat)
library(dplyr)
library(Seurat)
library(data.table)
library(future)
plan("multicore", workers = 12)
options(future.globals.maxSize = 10000 * 1024^10,
        future.rng.onMisuse = 'ignore')
sessionInfo()

Loading required package: Matrix


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, saveRDS


Loading Seurat v5 beta version 
To maintain compatibility with previous workflows, new Seurat objects will use the previous object structure by default
To use new Seurat v5 assays: Please run: options(Seurat.object.assay.version = 'v5')


Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last




R version 4.2.3 (2023-03-15)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Rocky Linux 8.7 (Green Obsidian)

Matrix products: default
BLAS/LAPACK: /gpfs/home3/cruiz2/miniconda3/envs/r_python/lib/libopenblasp-r0.3.21.so

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] future_1.32.0           data.table_1.14.8       Seurat_4.9.9.9040      
[4] SeuratObject_4.9.9.9081 sp_1.6-0                dplyr_1.1.1            
[7] numbat_1.2.3            Matrix_1.5-3           

loaded via a namespace (and not attached):
  [1] uuid_1.1-0             spam_

In [ ]:
dmg <- readRDS('/projects/0/einf2548/cruiz/dmg/data/merged_dmg_atlas_qc_filtered.rds')
dmg

An object of class Seurat 
38576 features across 409561 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

In [ ]:
dmg$numbat_id <- recode(dmg$predicted.annotation_level_3,
                         `AC-like` = 'Malignant',
                                       `MES-like` = "Malignant",    
                                       `OPC-like` = "Malignant", 
                                       `NPC-like` = "Malignant",    
                                       `Astrocyte` = "Glial_Neuronal", 
                                       # `Oligodendrocyte` = "Glial_Neuronal",    
                                       `OPC` = "Glial_Neuronal", 
                                       `Neuron` = "Glial_Neuronal",
                                        `Mono` = "Myeloid",
                                       `TAM-BDM` = "Myeloid",    
                                       `TAM-MG` = "Myeloid", 
                                       `DC` = "Myeloid",    
                                       `Mast` = "Myeloid",
                                       `CD4/CD8` = "Lymphoid",    
                                       `NK` = "Lymphoid", 
                                       `B cell` = "Lymphoid",    
                                       `Plasma B` = "Lymphoid",
                                       `Endothelial` = "Vascular",    
                                       `Mural cell` = "Vascular"
                         )
Idents(dmg) <- dmg$numbat_id
table(dmg@active.ident)


      Malignant  Glial_Neuronal        Vascular         Myeloid Oligodendrocyte 
         231886           78710            7188           60905           25955 
       Lymphoid 
           4917 

In [ ]:
`%nin%` <- function(x, y) !(x %in% y)
dmg <- subset(dmg, Study %nin% c('Filbin2018', 'Liu2022'))
dmg
gc()

An object of class Seurat 
38576 features across 397794 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,8382606,447.7,16414262,876.7,16414262,876.7
Vcells,12653726388,96540.3,16213889217,123702.2,12897181277,98397.7


In [ ]:
dmg <- subset(dmg, Batch_for_correction == '10Xv1_nuclei_multiome')
dmg

An object of class Seurat 
38576 features across 80155 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

In [ ]:
ref_internal = aggregate_counts(GetAssayData(subset(dmg, 
                                                   idents = c('Malignant', 'Glial_Neuronal'),
                                                   invert = TRUE),
                                             slot = 'counts'
                                            ), 
                               as.data.frame(dmg@active.ident) %>% 
                                `colnames<-`('group') %>% 
                                tibble::rownames_to_column('cell') %>%
                                filter(!group %in% c('Malignant', 'Glial_Neuronal'))
                               )
ref_internal

Warning message:
“The `slot` argument of `GetAssayData()` is deprecated as of SeuratObject 5.0.0.
ℹ Please use the `layer` argument instead.”


cell_dict
       Vascular         Myeloid Oligodendrocyte        Lymphoid 
            670            7673            6164             570 


,Vascular,Myeloid,Oligodendrocyte,Lymphoid
A1BG,6.903810e-06,1.908153e-05,2.231141e-05,1.850247e-05
A1CF,0.000000e+00,3.381536e-07,3.203988e-07,0.000000e+00
A2M,1.250926e-03,2.718272e-04,6.495358e-06,4.995666e-05
A2ML1,1.558925e-06,1.352614e-06,5.825433e-07,2.775370e-06
A3GALT2,2.227036e-07,9.178455e-07,3.786532e-07,0.000000e+00
A4GALT,3.251472e-05,9.661532e-07,2.038902e-07,0.000000e+00
A4GNT,2.227036e-07,0.000000e+00,5.825433e-08,0.000000e+00
AAAS,1.046707e-05,7.391072e-06,1.339850e-05,4.625616e-06
AACS,2.449739e-05,1.463722e-05,1.963171e-05,2.220296e-05
AADAC,1.336221e-06,0.000000e+00,0.000000e+00,0.000000e+00


### Process Jessa2022

In [ ]:
jessa <- SplitObject(dmg, split.by = 'SampleID')
jessa
gc()

$`P-1694_S-1694_Multiome`
An object of class Seurat 
38576 features across 5868 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

$`P-1701_S-1701_Multiome`
An object of class Seurat 
38576 features across 10566 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

$`P-1709_S-1709_Multiome`
An object of class Seurat 
38576 features across 19292 samples with

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,8139363,434.7,16414262,876.7,16414262,876.7
Vcells,8554342137,65264.5,16213889217,123702.2,15238403904,116259.8


In [ ]:
table(dmg$SampleID)


 P-1694_S-1694_Multiome  P-1701_S-1701_Multiome  P-1709_S-1709_Multiome 
                   5868                   10566                   19292 
 P-1764_S-1766_Multiome  P-1779_S-1781_Multiome  P-1780_S-1780_Multiome 
                   6799                    5173                    7868 
 P-6253_S-8498_Multiome  P-6337_S-8821_Multiome  P-6640_S-9581_Multiome 
                   3336                    7020                    4657 
P-6774_S-10146_Multiome 
                   9576 

In [ ]:
rm(dmg)
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,8138369,434.7,16414262,876.7,16414262,876.7
Vcells,7357113046,56130.4,16213889217,123702.2,15238403904,116259.8


In [ ]:
#check cell name nomenclature to add to numbat pipeline
head(colnames(jessa[[1]]))

[1] "multiome_P-1694_S-1694_GTTTGTAAGGCTGGCT-1"
[2] "multiome_P-1694_S-1694_GCTCTGGCAAATATCC-1"
[3] "multiome_P-1694_S-1694_CATAACGGTTGACTTC-1"
[4] "multiome_P-1694_S-1694_CACAGGCTCTTGACCC-1"
[5] "multiome_P-1694_S-1694_CCCTCACCAGTATGTT-1"
[6] "multiome_P-1694_S-1694_GCTGTGATCCGCAACA-1"

In [ ]:
length(jessa)

[1] 10

In [ ]:
names(jessa) <- gsub('_Multiome', '', names(jessa))
jessa

$`P-1694_S-1694`
An object of class Seurat 
38576 features across 5868 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

$`P-1701_S-1701`
An object of class Seurat 
38576 features across 10566 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

$`P-1709_S-1709`
An object of class Seurat 
38576 features across 19292 samples within 6 assays 
Active assay: 

In [ ]:
for(i in 1:length(jessa)) {
    cat(' ############################################\n',
        '### Numbat for dataset number ', names(jessa[i]), '###\n',
        '############################################\n')
    
    old <- Sys.time() # get start time
    
    file_path <- paste0('/projects/0/einf2548/cruiz/dmg/public_data/Jessa2022/EGAD00001008349_multiome/numbat_pileup_output_cellbender/',
                        names(jessa[i]),'/',names(jessa[i]),'_allele_counts.tsv.gz')
    
    if (file.exists(file_path)) {  # check if file exists
        df_allele = fread(file_path)
        df_allele$cell <- paste0('multiome_', names(jessa[i]),'_',df_allele$cell)
          
        # run
        out = run_numbat(
            as.matrix(GetAssayData(jessa[[i]], slot = 'counts', assay = 'RNA')), # gene x cell integer UMI count matrix 
            ref_internal, # reference expression profile, a gene x cell type normalized expression level matrix
            df_allele, # allele dataframe generated by pileup_and_phase script and MUST BE A DATATABLE!
            min_cells = 20,
            t = 1e-3,
            max_iter = 2,
            init_k = 3,
            ncores = 40,
            ncores_nni = 40,
            multi_allelic = TRUE,
            plot = TRUE,
            genome = "hg38",
            out_dir = paste0('/projects/0/einf2548/cruiz/dmg/notebooks/iCNV/numbat/dmg_atlas_segregated_ref/Jessa2022/multiome/',names(jessa[i]))
        )
      
        cat(' ############################################\n',
        '### Done with dataset', names(jessa[i]), '###\n',
        '############################################\n')
        
        # print elapsed time
        new <- Sys.time() - old # calculate difference
        print(new) # print in nice format
    } else {
        cat(' ############################################\n',
        '### Skipping dataset', names(jessa[i]), 'because the file does not exist ###\n',
        '############################################\n')
    }
}


 ############################################
 ### Numbat for dataset number  P-1694_S-1694 ###
 ############################################


Numbat version: 1.2.3
Running under parameters:
t = 0.001
alpha = 1e-04
gamma = 20
min_cells = 20
init_k = 3
max_cost = 1760.4
n_cut = 0
max_iter = 2
max_nni = 100
min_depth = 0
use_loh = auto
multi_allelic = TRUE
min_LLR = 5
min_overlap = 0.45
max_entropy = 0.5
skip_nj = FALSE
diploid_chroms = 
ncores = 40
ncores_nni = 40
common_diploid = TRUE
tau = 0.3
check_convergence = FALSE
plot = TRUE
genome = hg38
Input metrics:
5868 cells

Mem used: 61Gb

Approximating initial clusters using smoothed expression ..

Mem used: 61Gb

number of genes left: 12085

running hclust...

Iteration 1

Mem used: 19.2Gb

Running HMMs on 5 cell groups..

Expression noise level: low (0.45). 

Running HMMs on 3 cell groups..

Testing for multi-allelic CNVs ..

0 multi-allelic CNVs found: 

Evaluating CNV per cell ..

Mem used: 14.8Gb

All cells succeeded

Expanding allelic states..

No multi-allelic CNVs, skipping ..

No multi-allelic CNVs, skipping ..

No multi-allelic CNVs, skipping ..

Building phylogeny .

 ############################################
 ### Done with dataset P-1694_S-1694 ###
 ############################################
Time difference of 48.11133 mins
 ############################################
 ### Numbat for dataset number  P-1701_S-1701 ###
 ############################################


Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.5 GiB”
Numbat version: 1.2.3
Running under parameters:
t = 0.001
alpha = 1e-04
gamma = 20
min_cells = 20
init_k = 3
max_cost = 3169.8
n_cut = 0
max_iter = 2
max_nni = 100
min_depth = 0
use_loh = auto
multi_allelic = TRUE
min_LLR = 5
min_overlap = 0.45
max_entropy = 0.5
skip_nj = FALSE
diploid_chroms = 
ncores = 40
ncores_nni = 40
common_diploid = TRUE
tau = 0.3
check_convergence = FALSE
plot = TRUE
genome = hg38
Input metrics:
10566 cells

Mem used: 14.8Gb

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.1 GiB”
Approximating initial clusters using smoothed expression ..

Mem used: 14.8Gb

number of genes left: 12231

running hclust...

Iteration 1

Mem used: 26Gb

Running HMMs on 5 cell groups..

Expression noise level: medium (0.79). 

Running HMMs on 3 cell groups..

Testing for multi-allelic CNVs ..

0 multi-allelic CNVs found: 

Evaluating CNV per cell

 ############################################
 ### Done with dataset P-1701_S-1701 ###
 ############################################
Time difference of 2.110663 hours
 ############################################
 ### Numbat for dataset number  P-1709_S-1709 ###
 ############################################


...

 ############################################
 ### Done with dataset P-6337_S-8821 ###
 ############################################
Time difference of 1.048631 hours
 ############################################
 ### Numbat for dataset number  P-6640_S-9581 ###
 ############################################


Numbat version: 1.2.3
Running under parameters:
t = 0.001
alpha = 1e-04
gamma = 20
min_cells = 20
init_k = 3
max_cost = 1397.1
n_cut = 0
max_iter = 2
max_nni = 100
min_depth = 0
use_loh = auto
multi_allelic = TRUE
min_LLR = 5
min_overlap = 0.45
max_entropy = 0.5
skip_nj = FALSE
diploid_chroms = 
ncores = 40
ncores_nni = 40
common_diploid = TRUE
tau = 0.3
check_convergence = FALSE
plot = TRUE
genome = hg38
Input metrics:
4657 cells

Mem used: 12.9Gb

Approximating initial clusters using smoothed expression ..

Mem used: 12.9Gb

number of genes left: 12027

running hclust...

Iteration 1

Mem used: 17.4Gb

Running HMMs on 5 cell groups..

Expression noise level: high (1). Consider using a custom expression reference profile.

Running HMMs on 3 cell groups..

Testing for multi-allelic CNVs ..

0 multi-allelic CNVs found: 

Evaluating CNV per cell ..

Mem used: 14.7Gb

All cells succeeded

Expanding allelic states..

No multi-allelic CNVs, skipping ..

No multi-allelic CNVs, skipping ..

N

 ############################################
 ### Done with dataset P-6640_S-9581 ###
 ############################################
Time difference of 39.21305 mins
 ############################################
 ### Numbat for dataset number  P-6774_S-10146 ###
 ############################################


Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.4 GiB”
Numbat version: 1.2.3
Running under parameters:
t = 0.001
alpha = 1e-04
gamma = 20
min_cells = 20
init_k = 3
max_cost = 2872.8
n_cut = 0
max_iter = 2
max_nni = 100
min_depth = 0
use_loh = auto
multi_allelic = TRUE
min_LLR = 5
min_overlap = 0.45
max_entropy = 0.5
skip_nj = FALSE
diploid_chroms = 
ncores = 40
ncores_nni = 40
common_diploid = TRUE
tau = 0.3
check_convergence = FALSE
plot = TRUE
genome = hg38
Input metrics:
9576 cells

Mem used: 14.5Gb

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.0 GiB”
Approximating initial clusters using smoothed expression ..

Mem used: 14.5Gb

number of genes left: 11315

running hclust...

Iteration 1

Mem used: 23.7Gb

Running HMMs on 5 cell groups..

Expression noise level: medium (0.96). 

Running HMMs on 3 cell groups..

Testing for multi-allelic CNVs ..

0 multi-allelic CNVs found: 

Evaluating CNV per cel

 ############################################
 ### Done with dataset P-6774_S-10146 ###
 ############################################
Time difference of 1.922954 hours
